# FPT Clock C valuation and decision review

Information cutoff23November2026. Forward value31December2026E. This notebook
reproduces the valuation, its accounting scope, economic challenges and decision.
Saved outputs use a sequential ordinary-Python harness, not a Jupyter kernel.
The source/forecast walkthrough remains in02_three_statement_development.ipynb.


In [1]:
from pathlib import Path
import json,sys
from copy import deepcopy
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/equity_valuation').is_dir())
sys.path.insert(0,str(ROOT/'src'))
from equity_valuation.three_statement import run_forecast
from equity_valuation.clock_c import value_case,reverse_terminal_growth
opening=json.loads((ROOT/'data/intake/clock_c_opening_pending.json').read_text())
fc=json.loads((ROOT/'configs/forecast_research.json').read_text())
vc=json.loads((ROOT/'configs/valuation_clock_c.json').read_text())
forecasts={n:run_forecast(opening,c) for n,c in fc['scenarios'].items()}
values={n:value_case(f,vc['scenarios'][n],vc['common']) for n,f in forecasts.items()}
print('Value date:',vc['common']['valuation_date'],'Information cutoff:',vc['information_cutoff'])


Value date: 2026-12-31 Information cutoff: 2026-11-23


## 1 Forward value and financing
A policy shortfall remains visible. Cases are not probability intervals.

In [2]:
for n,v in values.items():
    print(n,'VND/share',round(v['fair_value_vnd'],2),'WACC',round(v['wacc']*100,2),'funding',v['financing_status'])
assert values['downside']['max_funding_gap']>0
assert values['base']['market_price_vnd'] is None


base VND/share 76156.29 WACC 11.69 funding FUNDED
downside VND/share 40318.55 WACC 13.19 funding POLICY_SHORTFALL
upside VND/share 109965.29 WACC 11.19 funding FUNDED


## 2 Trace the parent equity bridge
Associate book value and CIP are excluded from asset additions. Future cash is not added a second time.

In [3]:
v=values['base']
for k in ['operating_ev','associate_value','cash_net_of_reserve','investments','debt','nci_value','equity_value','diluted_shares_million']:
    print(k,round(v[k],3))
assert abs(v['checks']['equity_bridge'])<1e-6
print('Terminal share of EV:',round(v['terminal_share_of_ev']*100,2),'%')


operating_ev 89251.169
associate_value 47934.581
cash_net_of_reserve 4867.552
investments 21219.532
debt 16472.375
nci_value 1754.333
equity_value 145046.126
diluted_shares_million 1904.585
Terminal share of EV: 72.53 %


## 3 Operating drivers reach valuation
Stress collections in the existing forecast, then recompute the forward equity bridge and FCFF.

In [4]:
stress=deepcopy(fc['scenarios']['base'])
for p in stress['periods']:p['dso']+=30
sf=run_forecast(opening,stress)
sv=value_case(sf,vc['scenarios']['base'],vc['common'])
print('DSO plus30days fair value:',round(sv['fair_value_vnd'],2))
print('Base fair value:',round(v['fair_value_vnd'],2))
assert sv['fair_value_vnd']<v['fair_value_vnd']
assert sf['periods'][0]['balance_sheet']['cash']<forecasts['base']['periods'][0]['balance_sheet']['cash']


DSO plus30days fair value: 72683.9
Base fair value: 76156.29


## 4 Terminal challenges and hypothetical reverse DCF
Higher WACC reduces value. A solved hypothetical price is not an observed market expectation.

In [5]:
higher=value_case(forecasts['base'],vc['scenarios']['base'],vc['common'],wacc_override=v['wacc']+.01)
assert higher['fair_value_vnd']<v['fair_value_vnd']
print('WACC plus100bp value:',round(higher['fair_value_vnd'],2))
r=reverse_terminal_growth(forecasts['base'],vc['scenarios']['base'],vc['common'],v['fair_value_vnd'])
assert abs(r['terminal_growth']-.03)<1e-9
assert abs(r['residual_vnd'])<1e-6
print('Recovered base g:',r['terminal_growth'],'Residual VND:',r['residual_vnd'])


WACC plus100bp value: 71149.56
Recovered base g: 0.029999999999999923 Residual VND: -1.4551915228366852e-11


## 5 Decision test
Do not replace the cash gate with a mechanical valuation rating.

In [6]:
reported=json.loads((ROOT/'data/intake/q3_2026_reported_schedules.json').read_text())
cfo9=next(g['reported_total_vnd']/1e9 for g in reported['groups'] if g['id']=='cfo')
q=forecasts['base']['periods'][0]
conversion=(cfo9+q['cash_flow']['cfo'])/(fc['actuals']['parent_pat_9m']+q['income_statement']['parent_pat'])
print('FY2026E CFO / parent PAT:',round(conversion*100,2),'%')
print('Cash gate:', 'PASS' if conversion>=.30 else 'FAIL')
print('Research stance: WATCHLIST. Same-cutoff quote is unavailable.')
assert conversion<.30


FY2026E CFO / parent PAT: 26.26 %
Cash gate: FAIL
Research stance: WATCHLIST. Same-cutoff quote is unavailable.


## 6 Independent engine receipts
These receipts describe numerical agreement, not empirical validation of assumptions.

In [7]:
out=ROOT/'outputs/p1_final'
excel=json.loads((out/'excel_valuation_reconciliation.json').read_text())
js=json.loads((out/'javascript_valuation_reconciliation.json').read_text())
assert js['all_pass']
assert all(abs(r['excel']-r['python'])<1e-6 for r in excel['comparisons'])
print('Excel valuation comparisons:',len(excel['comparisons']))
print('Excel native sensitivity points:',excel['native_sensitivity_points'])
print('JavaScript valuation comparisons:',js['numeric_comparisons'])


Excel valuation comparisons: 15
Excel native sensitivity points: 75
JavaScript valuation comparisons: 147
